In [ ]:
# Colab setup: install dependencies
!pip -q install xarray netCDF4 cftime cartopy

# Download the monthly GPCP netCDF (≈20 MB; 2.5° global grid, 1979–present)
# Source directory listing shows 'precip.mon.mean.nc' here:
# https://downloads.psl.noaa.gov/Datasets/gpcp/
!wget -q -N https://downloads.psl.noaa.gov/Datasets/gpcp/precip.mon.mean.nc

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# Open the dataset
ds = xr.open_dataset("precip.mon.mean.nc", decode_times=False)
print(ds)  # This shows you the meta information of this dataset, take a look at the Coordinates and Attributes


In [ ]:
# Read the variables & you can uncomment the command below to print
time = ds['time'].values          # days since 1800-01-01 00:00:00
lat  = ds['lat'].values           # -88.75..88.75
lon  = ds['lon'].values           # 1.25..358.75
ppt  = ds['precip']               # (time, lat, lon) in mm/day

# print(ds['time'])
# print(ds['lat'])
# print(ds['lon'])
# print(ds['precip'])


In [ ]:
# Start date
start_date = pd.Timestamp('1800-01-01')

# Convert to pandas datetime64 (vectorized)
new_date = start_date + pd.to_timedelta(time, unit='D')

# Quick checks on time:
print(time[0])        # e.g., 65378 days since 1800-01-01 00:00:00
print(new_date[0])    # human-readable date, date for the first map
print(new_date[-1])   # date for the last map in the dataset


In [ ]:
# This section will generate two maps: precipitation for January 1979 & July 1979

def plot_precip(data, lon, lat, title, vmin=0, vmax=16):
    #1: specify projection info
    map_crs = ccrs.Robinson(central_longitude=180.)
    data_crs = ccrs.PlateCarree()

    #2: create figure and axes objects
    fig = plt.figure(figsize=(10,5))
    ax = fig.add_subplot(111, projection=map_crs)

    #3: add coastlines
    ax.coastlines(resolution='50m') # res can be '110m', '50m', or '10m'

    #3: add contour level information and plot
    contour_levels = np.arange(vmin,vmax,1)

    contour_plot = ax.contourf(lon,
                              lat,
                              data,
                              cmap='YlGnBu',
                              levels=contour_levels,
                              transform=data_crs,
                              extend='max')

    contour_lines = ax.contour(lon,
                              lat,
                              data,
                              levels=contour_levels,
                              transform=data_crs,
                              linewidths=[0.5],
                              colors=['0.5'])

    # add final touches:  color bars, titles
    cbar = fig.colorbar(contour_plot, orientation='horizontal', fraction=.05, aspect=30, pad=.05)
    cbar.set_label('mm day$^{-1}$')
    ax.set_title(title)

plot_precip(ppt.isel(time=0),lon,lat,title='Precipitation of January 1979 (GPCP)')
plot_precip(ppt.isel(time=6),lon,lat,title='Precipitation of July 1979 (GPCP)')

In [ ]:
# This section compute the precipitation difference between 2020-2014 and 1979-1983
ppt_dt = ppt.assign_coords(time=new_date)

# Select periods (inclusive)
ppt_7983 = ppt_dt.sel(time=slice('1979-01-01', '1983-12-31'))
ppt_2024 = ppt_dt.sel(time=slice('2020-01-01', '2024-12-31'))

# Means over time for each period
prep_avg7983 = ppt_7983.mean(dim='time', skipna=True)
prep_avg2024 = ppt_2024.mean(dim='time', skipna=True)

# Difference (mm/day): 2020–2024 minus 1979–1983
dprep = (prep_avg2024 - prep_avg7983)


In [ ]:
#1: specify projection info
map_crs = ccrs.Robinson(central_longitude=180.)
data_crs = ccrs.PlateCarree()

#2: create figure and axes objects
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(111, projection=map_crs)

#3: add coastlines
ax.coastlines(resolution='50m') # res can be '110m', '50m', or '10m'

#3: add contour level information and plot
contour_levels = np.arange(-2,2,0.2)

contour_plot = ax.contourf(lon,
                           lat,
                           dprep,
                           cmap='twilight_shifted_r',
                           levels=contour_levels,
                           transform=data_crs,
                           extend='both')

contour_lines = ax.contour(lon,
                           lat,
                           dprep,
                           levels=contour_levels,
                           transform=data_crs,
                           linewidths=[0.5],
                           colors=['0.5'])

# add final touches:  color bars, titles
cbar = fig.colorbar(contour_plot, orientation='horizontal', fraction=.05, aspect=30, pad=.05)
cbar.set_label('mm day$^{-1}$')
ax.set_title('Precipitation (2020–2024 minus 1979–1983), GPCP')